# PigeonPilot — Interactive Playground

Load a **finished** named run from `Models.ipynb` (Step 10), review its test metrics, then draw a displacement route and watch both pigeons fly it and read out a home direction.

Requires: kernel **Python 3.11**, `anywidget` (`pip install anywidget`), and at least one run under `outputs/checkpoints/`.

Change the `RUN_NAME` parameter to import different models from `/outputs/checkpoints`.


### 1. Imports + pick a run


In [1]:
from IPython.display import Markdown, display

from resources.snn import list_runs, load_run

runs = list_runs()
assert runs, "No checkpoints yet — run Models.ipynb Step 10 (save_run) first."

display(Markdown("### Available runs"))
for r in runs:
    mark = " ← latest" if r["is_latest"] else ""
    display(Markdown(f"- `{r['name']}` · n_res={r['n_reservoir']}{mark}"))

# Change this to a concrete name, e.g. "n10000_main", or keep "latest"
RUN_NAME = "n1000_main"
print("will load:", RUN_NAME)


### Available runs

- `n10_main` · n_res=10 ← latest

- `n50_main` · n_res=50

- `n10000_main` · n_res=10000

- `n1000_main` · n_res=1000

- `n100_main` · n_res=100

will load: n1000_main


### 2. Load checkpoint + show metrics


In [ ]:
bundle = load_run(RUN_NAME)
cfg = bundle.config
metrics = bundle.metrics

display(Markdown(
    f"### Loaded `{RUN_NAME}`\n"
    f"- reservoir size: **{cfg.n_reservoir}**\n"
    f"- encoding: v=`{cfg.encoding_velocity:.4f}`, dt=`{cfg.encoding_dt}`, "
    f"rate=`{cfg.input_rate_hz}` Hz, silence=`{cfg.trailing_silence}`\n"
    f"- ridge α=`{cfg.ridge_alpha}`"
))

summary = metrics.get("summary") or {}
if summary:
    lines = ["### Test angular error (from training run)\n"]
    for name in ("A", "B"):
        if name not in summary:
            continue
        s = summary[name]
        lines.append(
            f"- **Pigeon {name}**: mean error {s['mean_deg']:.1f}° ± {s['std_deg']:.1f}° "
            f"| exact-bin acc {100 * s['exact_acc']:.1f}%"
        )
    by_diff = metrics.get("by_difficulty") or {}
    if by_diff:
        lines.append("\n| difficulty | A mean ° | B mean ° | n |")
        lines.append("|---|---:|---:|---:|")
        for diff, row in by_diff.items():
            lines.append(
                f"| {diff} | {row['A_mean_deg']:.1f} | {row['B_mean_deg']:.1f} | {row['n']} |"
            )
    display(Markdown("\n".join(lines)))
else:
    display(Markdown("_No metrics stored in this checkpoint — demo still works._"))


### 3. Draw → displace → predict

1. **Draw** a route with the mouse, starting at the green home star.
2. Click **Fly**. The bird flies your route while the spike raster below fills in *live* — each tick is the body-ring neuron that points North on that leg.
3. At the release point both pigeons read out a home direction and fly it **side by side**: A (fixed reservoir) and B (STDP). The blue dashed line is the true home vector.
4. The rings show each model's full 36-bin direction profile.

In [4]:
from resources.widget import PigeonPlayground

# One canvas, both pigeons; set models=("A",) to run a single pigeon.
# reference=True scores the 147 held-out routes once (~8 s, then cached next to
# the checkpoint) so every flight can be shown against the error distribution
# it was drawn from.
playground = PigeonPlayground(bundle, models=("A", "B"), reference=True)
playground
